In [ ]:
"""LiDAR 点群からの地面高さ推定アルゴリズムの検証スクリプト.

TypeScript 実装 (frontend/src/lib/groundHeight.ts の estimateGroundZ /
estimateGroundZGlobal) と同一ロジックの Python 移植版.
実データに対して複数の位置で推定を実行し、結果を可視化して妥当性を確認する.

センサー座標系とグローバル座標系には傾き (LiDAR 取り付け回転 + ego のピッチ/ロール)
があるため、z 方向の変換にスカラー加算 (ground_z + sensor_height + ego_z) を使うと
傾きに比例した誤差が target の ego からの距離に応じて増大する。
よって z 方向の変換には ego pose と calibrated sensor による厳密な剛体変換
(global_to_sensor / sensor_to_global) を使う。

依存: requests, numpy, matplotlib, pyyaml
    pip install requests numpy matplotlib pyyaml
"""
from pathlib import Path

import requests
import numpy as np
import matplotlib.pyplot as plt

# ── 設定 ─────────────────────────────────────────────────────────────────────

API_HOST = 'http://192.168.11.7'
API_PORT = 8000
SAMPLE_TOKEN = '3e8750f331d7499e9b5123e9eb70f2e2'

# アルゴリズムパラメータ (frontend/config/settings.yml annotation.ground_height_detection と同期)
_SETTINGS_YML = Path('../frontend/config/settings.yml')
try:
    import yaml
    _p = yaml.safe_load(_SETTINGS_YML.read_text())['annotation']['ground_height_detection']
    SEARCH_RADII      = [float(r) for r in _p['search_radii']]  # 半径探索の段階 (m)
    MIN_POINTS        = int(_p['min_points'])                   # 有効とみなす最低点数
    BIN_SIZE          = float(_p['bin_size'])                   # ヒストグラムビン幅 (m)
    REFINE_HALF_WIDTH = float(_p['refine_half_width'])          # 精密化時にビン中心から拾う半幅 (m)
    LOWER_MARGIN      = float(_p['lower_margin'])               # prior より下に許容する範囲 (m)
    UPPER_MARGIN      = float(_p['upper_margin'])               # prior より上に許容する範囲 (m)
    MAX_DEVIATION     = float(_p['max_deviation'])              # prior からの最大許容乖離 (m)
    print(f'パラメータを {_SETTINGS_YML} から読み込みました')
except Exception as e:
    # settings.yml が読めない環境向けフォールバック (2026-07 時点の settings.yml と同値)
    print(f'settings.yml 読み込み失敗 ({e}) → ハードコード値を使用')
    SEARCH_RADII      = [2.0, 4.0, 6.0]
    MIN_POINTS        = 20
    BIN_SIZE          = 0.1
    REFINE_HALF_WIDTH = 0.2
    LOWER_MARGIN      = 1.0
    UPPER_MARGIN      = 1.0
    MAX_DEVIATION     = 1.0

# ── 座標変換 ──────────────────────────────────────────────────────────────────
# TypeScript coordinateUtils.ts の quatToRotMat / globalToSensor / sensorToGlobal と同一式

def quat_to_rot(q) -> np.ndarray:
    """クォータニオン [w, x, y, z] → 3×3 回転行列"""
    w, x, y, z = q
    return np.array([
        [1 - 2*(y*y + z*z), 2*(x*y - z*w),     2*(x*z + y*w)],
        [2*(x*y + z*w),     1 - 2*(x*x + z*z), 2*(y*z - x*w)],
        [2*(x*z - y*w),     2*(y*z + x*w),     1 - 2*(x*x + y*y)],
    ])


def global_to_sensor(p, ego_pose: dict, calib_sensor: dict) -> np.ndarray:
    """グローバル座標系 → センサー座標系 (厳密剛体変換)"""
    p_ego = quat_to_rot(ego_pose['rotation']).T @ (np.asarray(p, dtype=np.float64) - np.asarray(ego_pose['translation']))
    return quat_to_rot(calib_sensor['rotation']).T @ (p_ego - np.asarray(calib_sensor['translation']))


def sensor_to_global(p, ego_pose: dict, calib_sensor: dict) -> np.ndarray:
    """センサー座標系 → グローバル座標系 (global_to_sensor の逆変換)"""
    p_ego = quat_to_rot(calib_sensor['rotation']) @ np.asarray(p, dtype=np.float64) + np.asarray(calib_sensor['translation'])
    return quat_to_rot(ego_pose['rotation']) @ p_ego + np.asarray(ego_pose['translation'])


# ── アルゴリズム本体 ──────────────────────────────────────────────────────────

def estimate_ground_z(
    points: np.ndarray,
    target_xy: tuple[float, float],
    prior_z: float,
    verbose: bool = False,
) -> tuple[float, float] | None:
    """指定位置周辺の地面高さ (センサー座標系 z) を推定する.

    TypeScript groundHeight.ts estimateGroundZ と同一のコア検出.

    Args:
        points:    (N, 3+) 点群 [x, y, z, ...] (LiDAR センサー座標系)
        target_xy: 地面高さを求めたい水平位置 (x, y) (センサー座標系)
        prior_z:   その位置で予測される地面のセンサー座標 z.
                   zフィルタ窓 [prior_z - LOWER_MARGIN, prior_z + UPPER_MARGIN] の中心と
                   MAX_DEVIATION 判定の基準に使う (ego 直下では ≈ -LiDAR取り付け高さ)
        verbose:   途中経過を表示

    Returns:
        (ground_z_sensor, radius): 推定値 (センサー座標系) と採用半径。
        全半径で条件を満たさなければ None
    """
    z_min = prior_z - LOWER_MARGIN
    z_max = prior_z + UPPER_MARGIN
    tx, ty = target_xy

    # 事前 z フィルタ (全半径で共通なので先に 1 回だけ)
    z_ok = (points[:, 2] >= z_min) & (points[:, 2] <= z_max)
    pts = points[z_ok]

    dx = pts[:, 0] - tx
    dy = pts[:, 1] - ty
    dist2 = dx * dx + dy * dy

    bin_count = int(np.ceil((z_max - z_min) / BIN_SIZE - 1e-9))

    for radius in SEARCH_RADII:
        zs = pts[dist2 <= radius * radius, 2]

        # ヒストグラム化し、点数最大のビンを選択 (同数タイは低い方)
        bins, bin_edges = np.histogram(zs, bins=bin_count, range=(z_min, z_max))
        ground_bin_idx = int(np.argmax(bins))

        # ビン中心 ±REFINE_HALF_WIDTH の点の中央値で精密化
        bin_center = z_min + (ground_bin_idx + 0.5) * BIN_SIZE
        refined = zs[np.abs(zs - bin_center) <= REFINE_HALF_WIDTH]
        if verbose:
            print(f'  R={radius}m: {len(refined)} 点 (精密化後)')
        # 最低点数の妥当性チェック
        if len(refined) < MIN_POINTS:
            continue  # 半径を拡大して再試行

        ground_z = float(np.median(refined))

        if verbose:
            print(f'  選択ビン: [{bin_edges[ground_bin_idx]:.2f}, '
                  f'{bin_edges[ground_bin_idx + 1]:.2f}] ({bins[ground_bin_idx]} 点)')
            print(f'  精密化: {len(refined)} 点の median = {ground_z:.3f}')

        # prior からの乖離妥当性チェック
        if abs(ground_z - prior_z) > MAX_DEVIATION:
            continue  # prior から乖離しすぎ → 次の半径で再試行

        return ground_z, radius

    return None


def estimate_ground_z_global(
    points: np.ndarray,
    target_global_xy: tuple[float, float],
    ego_pose: dict,
    calib_sensor: dict,
    verbose: bool = False,
) -> tuple[float, str]:
    """グローバル座標系の (x, y) に対する地面高さ推定.

    TypeScript groundHeight.ts estimateGroundZGlobal と同一.
    グローバル (x, y, ego_z) (= ego と同じ高さの地面点) を global_to_sensor で厳密変換し、
    その z を prior としてセンサー座標系で検出。検出値は sensor_to_global で厳密に
    グローバル座標へ戻すため、座標系間に傾きがあっても target の距離に依らず正確.

    Returns:
        (ground_z_global, method): グローバル座標系の地面高さと判定経路のラベル。
        検出できなければ ego_z を返す
    """
    ego_z = ego_pose['translation'][2]
    sp = global_to_sensor([target_global_xy[0], target_global_xy[1], ego_z], ego_pose, calib_sensor)
    result = estimate_ground_z(points, (sp[0], sp[1]), sp[2], verbose)  # sp[2] = 傾き補正済み prior
    if result is None:
        return ego_z, 'prior(fallback)'
    ground_z_sensor, radius = result
    ground_z_global = float(sensor_to_global([sp[0], sp[1], ground_z_sensor], ego_pose, calib_sensor)[2])
    return ground_z_global, f'estimated(R={radius})'


# ── データ取得 ────────────────────────────────────────────────────────────────

def fetch_pointcloud(url: str) -> np.ndarray:
    print(f'点群取得中: {url}')
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    points = np.array(resp.json()['points'], dtype=np.float64)
    print(f'取得完了: {len(points)} 点')
    return points


# ── 検証 ─────────────────────────────────────────────────────────────────────

def main() -> None:
    api = f'{API_HOST}:{API_PORT}/api/v1'
    sensor_data = requests.get(f'{api}/samples/{SAMPLE_TOKEN}/sensor-data').json()['LIDAR_TOP']
    # 点群フレームと一致する LIDAR_TOP の ego_pose (translation + rotation) を使う
    ego_pose = sensor_data['ego_pose']
    ego_x, ego_y, ego_z = ego_pose['translation']
    print(f'ego_pose.translation: ({ego_x:.3f}, {ego_y:.3f}, {ego_z:.3f})')
    calib_sensor = requests.get(f'{api}/calibrated-sensors/{sensor_data["calibrated_sensor_token"]}').json()
    lidar_height = calib_sensor['translation'][2]
    print(f'LIDAR_TOP の取り付け高さ (calibrated_sensor.translation[2]): {lidar_height:.3f} m')
    points = fetch_pointcloud(f'{api}/sensor-data/{sensor_data["token"]}/pointcloud')

    # pointcloudのzの生値のヒストグラムをプロット
    plt.figure(figsize=(10, 5))
    plt.hist(points[:, 2], bins=100, color='blue', alpha=0.7)
    plt.title('Raw Point Cloud Z Histogram')
    plt.xlabel('Z (m)')
    plt.ylabel('Frequency')
    plt.xlim(-5, 5)  # zの範囲を制限して見やすくする
    plt.show()

    # ── 検証 1: 複数のターゲット位置で推定 (グローバル座標、ego からのオフセットで指定) ──
    # 自車近傍 / 前後左右 / 遠方 を含むグリッド
    offsets = [
        (0.0,   0.0),    # 自車直下
        (5.0,   0.0),
        (-5.0,  0.0),
        (0.0,   5.0),
        (10.0,  10.0),
        (20.0,  0.0),    # 遠方
        (30.0,  0.0),    # さらに遠方 (点が疎な可能性)
        (40.0,  40.0),   # おそらく点が無い → prior フォールバック確認
    ]

    print('\n── 各ターゲット位置での推定結果 ──')
    print(f'{"offset (x, y)":>16} | {"ground_z":>9} | method')
    print('-' * 55)
    results = []
    for ox, oy in offsets:
        gxy = (ego_x + ox, ego_y + oy)
        gz, method = estimate_ground_z_global(points, gxy, ego_pose, calib_sensor)
        results.append((gxy, gz, method))
        print(f'({ox:6.1f}, {oy:6.1f}) | {gz:9.4f} | {method}')

    # ── 検証 2: 自車直下の詳細ログ ─────────────────────────────────────────────
    print('\n── 自車直下 (ego位置) の詳細 ──')
    estimate_ground_z_global(points, (ego_x, ego_y), ego_pose, calib_sensor, verbose=True)

    # ── 検証 3: 可視化 ────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # 自車直下の prior (グローバル (ego_x, ego_y, ego_z) をセンサー座標に厳密変換した z)
    sp0 = global_to_sensor([ego_x, ego_y, ego_z], ego_pose, calib_sensor)
    prior0 = float(sp0[2])

    # 3-1: BEV (上から見た点群、センサー座標系) + ターゲット位置
    ax = axes[0]
    # 描画負荷軽減のため間引き
    step = max(1, len(points) // 30000)
    sub = points[::step]
    sc = ax.scatter(sub[:, 0], sub[:, 1], c=sub[:, 2], s=0.5,
                    cmap='viridis', vmin=prior0 - 0.5, vmax=prior0 + 2.5)
    for (gxy, gz, method) in results:
        sp = global_to_sensor([gxy[0], gxy[1], ego_z], ego_pose, calib_sensor)
        color = 'red' if method.startswith('estimated') else 'orange'
        ax.plot(sp[0], sp[1], 'x', color=color, markersize=10, markeredgewidth=2)
    ax.set_title('BEV (color = z) / x=estimated, orange=prior fallback')
    ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
    ax.set_aspect('equal')
    plt.colorbar(sc, ax=ax, label='z (m)')

    # 3-2: 自車近傍の z ヒストグラム (センサー座標系)
    # LiDAR 直下は近接ブラインドで点が無いことがあるため、ego 直下推定で採用された半径を使う
    ax = axes[1]
    z_min = prior0 - LOWER_MARGIN
    z_max = prior0 + UPPER_MARGIN
    est0 = estimate_ground_z(points, (sp0[0], sp0[1]), prior0)
    r0 = est0[1] if est0 is not None else SEARCH_RADII[-1]
    d2 = (points[:, 0] - sp0[0]) ** 2 + (points[:, 1] - sp0[1]) ** 2
    near = points[(d2 <= r0 * r0) & (points[:, 2] >= z_min) & (points[:, 2] <= z_max)]
    ax.hist(near[:, 2], bins=int(np.ceil((z_max - z_min) / BIN_SIZE - 1e-9)), range=(z_min, z_max),
            edgecolor='black', alpha=0.7)
    if est0 is not None:
        ax.axvline(est0[0], color='red', linestyle='--', label=f'estimated = {est0[0]:.3f}')
    ax.axvline(prior0, color='orange', linestyle=':', label=f'prior = {prior0:.2f}')
    ax.set_title(f'z histogram near ego (R={r0}m, sensor frame)')
    ax.set_xlabel('z (m)'); ax.set_ylabel('count')
    ax.legend()

    # 3-3: 側面図 (センサー座標系 x-z 断面, |y| < 2m の点) + 推定地面ライン
    # センサー x ごとに対応するグローバル (x, y) を求めて厳密変換版で推定し、
    # 結果のグローバル地面点をセンサー座標に戻して同一フレームで描画する
    ax = axes[2]
    slice_pts = points[np.abs(points[:, 1]) < 2.0]
    ax.scatter(slice_pts[:, 0], slice_pts[:, 2], s=0.5, alpha=0.5)
    line_x, line_z = [], []
    for x in np.arange(-30, 31, 2.0):
        g = sensor_to_global([float(x), 0.0, prior0], ego_pose, calib_sensor)
        gz, _ = estimate_ground_z_global(points, (g[0], g[1]), ego_pose, calib_sensor)
        s = global_to_sensor([g[0], g[1], gz], ego_pose, calib_sensor)
        line_x.append(s[0]); line_z.append(s[2])
    ax.plot(line_x, line_z, 'r.-', label='estimated ground')
    ax.axhline(prior0, color='orange', linestyle=':', label='prior (at ego)')
    ax.set_title('side view (|y| < 2m, sensor frame) + estimated ground line')
    ax.set_xlabel('x (m)'); ax.set_ylabel('z (m)')
    ax.set_ylim(prior0 - 1.5, prior0 + 4.0)
    ax.legend()

    plt.tight_layout()
    out_path = 'ground_estimation_check.png'
    plt.savefig(out_path, dpi=120)
    print(f'\n可視化を保存: {out_path}')
    plt.show()


if __name__ == '__main__':
    main()